# Data Processing

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os
import warnings

import matplotlib.dates as mdates
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================================
# JIDT Setup
# ============================================================================
try:
    from jpype import *
except ImportError:
    print("ERROR: jpype1 is required. Install with: pip install jpype1")
    raise

try:
    import yfinance as yf
except ImportError:
    print("ERROR: yfinance is required. Install with: pip install yfinance")
    raise

# ============================================================================
# CONFIGURE THIS: Path to infodynamics.jar
# Download JIDT from: https://github.com/jlizier/jidt/releases
# ============================================================================
jar_location = "infodynamics.jar"   # Update to your local path, e.g. "/path/to/infodynamics.jar"

if not isJVMStarted():
    if not os.path.isfile(jar_location):
        raise FileNotFoundError(
            f"infodynamics.jar not found at '{jar_location}'.\n"
            "Download JIDT from https://github.com/jlizier/jidt/releases "
            "and update the jar_location variable above."
        )
    startJVM(getDefaultJVMPath(), "-ea", "-Djava.class.path=" + jar_location, convertStrings=True)
    print("✓ JIDT library loaded")
else:
    print("✓ JIDT library already loaded")

In [4]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Australian bank tickers
BANKS = {
    'CBA': 'CBA.AX',      # Commonwealth Bank
    'WBC': 'WBC.AX',      # Westpac
    'ANZ': 'ANZ.AX',      # ANZ
    'NAB': 'NAB.AX',      # National Australia Bank
    'MQG': 'MQG.AX'       # Macquarie Group
}

# Control variables (market factors)
CONTROLS = {
    'Interest_Rates': 'DE000SLA6QZ3.SG',  # SPDR S&P/ASX iBoxx Australian Government Bond
    'ASX200': 'DE000SL0CK81.SG'   # ASX200 index excluding top 20 stocks
}

bank_names = list(BANKS.keys())
control_names = list(CONTROLS.keys())

# Analysis parameters
K_NEIGHBORS = 4                   # KSG estimator neighbors
ALGORITHM = 'ksg1'                # KSG algorithm ('ksg1' or 'ksg2')
WINDOW_SIZE = 150                 # Sliding window size (days)
START_DATE = '2014-01-01'         # Analysis start date
END_DATE = '2025-11-01'           # Analysis end date

print("Configuration loaded ✓")


Configuration loaded ✓


In [5]:
# ============================================================================
# DOWNLOAD STOCK DATA
# ============================================================================

all_tickers = {**BANKS, **CONTROLS}
data = {}

for name, ticker in all_tickers.items():
    df = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
    adj_close = df['Adj Close'].iloc[:, 0] if 'Adj Close' in df.columns.get_level_values(0) else df['Close'].iloc[:, 0]
    data[name] = adj_close

prices = pd.DataFrame(data).fillna(method='ffill', limit=5).dropna()
print(f"✓ Downloaded {len(prices)} days for {len(prices.columns)} assets")

# ============================================================================
# CALCULATE LOG RETURNS
# ============================================================================

returns = np.log(prices / prices.shift(1)).dropna()

# Standardize returns (z-score normalization: mean=0, std=1)
# Important for KSG estimator which is sensitive to scale
returns = (returns - returns.mean()) / returns.std()

✓ Downloaded 3037 days for 7 assets


In [6]:
# ============================================================================
# FUNCTIONS - TIME SERIES FOCUSED
# ============================================================================


def calculate_mi_time_series(data_df, bank_list, window_size=100, 
                              k_neighbors=4, theiler_window=0, algorithm='ksg1',
                              num_surrogates=0):
    """
    Calculate MI time series using sliding window with optional significance testing.
    
    Args:
        num_surrogates: Number of surrogate datasets for significance testing (0 = no testing)
    
    Returns:
        mi_ts_df: DataFrame with MI time series for all bank pairs (excluding diagonal)
        pval_ts_df: DataFrame with p-values (only if num_surrogates > 0, else None)
    """
    # Initialize storage
    mi_series = {f"{b1}-{b2}": [] for i, b1 in enumerate(bank_list) 
                 for j, b2 in enumerate(bank_list) if i < j}
    pval_series = {f"{b1}-{b2}": [] for i, b1 in enumerate(bank_list) 
                   for j, b2 in enumerate(bank_list) if i < j} if num_surrogates > 0 else None
    dates = []
    
    # Get JIDT class
    MIClass = (JPackage('infodynamics.measures.continuous.kraskov').MutualInfoCalculatorMultiVariateKraskov1 
               if algorithm == 'ksg1' else 
               JPackage('infodynamics.measures.continuous.kraskov').MutualInfoCalculatorMultiVariateKraskov2)
    
    # Sliding window
    for end_idx in range(window_size, len(data_df) + 1):
        window_data = data_df.iloc[end_idx - window_size:end_idx]
        dates.append(data_df.index[end_idx - 1])
        
        # Calculate MI for all pairs
        for i, bank1 in enumerate(bank_list):
            for j, bank2 in enumerate(bank_list):
                if i < j:
                    X = window_data[bank1].values.flatten()
                    Y = window_data[bank2].values.flatten()
                    
                    calc = MIClass()
                    calc.initialise(1, 1)
                    calc.setProperty("k", str(k_neighbors))
                    if theiler_window > 0:
                        calc.setProperty("DYN_CORR_EXCL", str(theiler_window))
                    calc.setProperty("NOISE_LEVEL_TO_ADD", "1e-8")
                    calc.setObservations(JArray(JDouble, 1)(X.tolist()), 
                                        JArray(JDouble, 1)(Y.tolist()))
                    
                    # Calculate MI
                    mi_value = calc.computeAverageLocalOfObservations()
                    mi_series[f"{bank1}-{bank2}"].append(mi_value)
                    
                    # Calculate significance if requested
                    if num_surrogates > 0:
                        significance = calc.computeSignificance(num_surrogates)
                        pval_series[f"{bank1}-{bank2}"].append(significance.pValue)
    
    mi_ts_df = pd.DataFrame(mi_series, index=dates)
    pval_ts_df = pd.DataFrame(pval_series, index=dates) if num_surrogates > 0 else None
    
    return (mi_ts_df, pval_ts_df) if num_surrogates > 0 else mi_ts_df


def calculate_cmi_time_series(data_df, bank_list, control_vars, window_size=100, 
                               k_neighbors=4, theiler_window=0, algorithm='ksg1',
                               num_surrogates=0):
    """
    Calculate CMI time series using sliding window with optional significance testing.
    
    Args:
        data_df: DataFrame with standardized returns
        bank_list: List of bank column names
        control_vars: List of control variable column names OR DataFrame slice
        window_size: Sliding window size
        k_neighbors: Number of neighbors for KSG
        theiler_window: Theiler window for temporal exclusion
        algorithm: 'ksg1' or 'ksg2'
        num_surrogates: Number of surrogate datasets for significance testing (0 = no testing)
    
    Returns:
        cmi_ts_df: DataFrame with CMI time series for all bank pairs
        pval_ts_df: DataFrame with p-values (only if num_surrogates > 0, else None)
    """
    # Initialize storage
    cmi_series = {f"{b1}-{b2}": [] for i, b1 in enumerate(bank_list) 
                  for j, b2 in enumerate(bank_list) if i < j}
    pval_series = {f"{b1}-{b2}": [] for i, b1 in enumerate(bank_list) 
                   for j, b2 in enumerate(bank_list) if i < j} if num_surrogates > 0 else None
    dates = []
    
    # Get JIDT class
    CMIClass = (JPackage('infodynamics.measures.continuous.kraskov').ConditionalMutualInfoCalculatorMultiVariateKraskov1 
                if algorithm == 'ksg1' else 
                JPackage('infodynamics.measures.continuous.kraskov').ConditionalMutualInfoCalculatorMultiVariateKraskov2)
    
    # Determine number of conditioning dimensions
    if isinstance(control_vars, list):
        n_cond_dims = len(control_vars)
    else:
        n_cond_dims = 1 if len(control_vars.shape) == 1 else control_vars.shape[1]
    
    # Sliding window
    for end_idx in range(window_size, len(data_df) + 1):
        window_data = data_df.iloc[end_idx - window_size:end_idx]
        dates.append(data_df.index[end_idx - 1])
        
        # Get conditioning variables
        if isinstance(control_vars, list):
            Z = window_data[control_vars].values
        else:
            Z = window_data[control_vars].values.reshape(-1, 1)
        Z_java = JArray(JDouble, 2)(Z.tolist())
        
        # Calculate CMI for off-diagonal pairs
        for i, bank1 in enumerate(bank_list):
            for j, bank2 in enumerate(bank_list):
                if i < j:
                    X = window_data[bank1].values.flatten()
                    Y = window_data[bank2].values.flatten()
                    
                    calc = CMIClass()
                    calc.initialise(1, 1, n_cond_dims)
                    calc.setProperty("k", str(k_neighbors))
                    if theiler_window > 0:
                        calc.setProperty("DYN_CORR_EXCL", str(theiler_window))
                    calc.setProperty("NOISE_LEVEL_TO_ADD", "1e-8")
                    calc.setObservations(JArray(JDouble, 1)(X.tolist()), 
                                        JArray(JDouble, 1)(Y.tolist()), Z_java)
                    
                    # Calculate CMI
                    cmi_value = calc.computeAverageLocalOfObservations()
                    cmi_series[f"{bank1}-{bank2}"].append(cmi_value)
                    
                    # Calculate significance if requested
                    if num_surrogates > 0:
                        significance = calc.computeSignificance(num_surrogates)
                        pval_series[f"{bank1}-{bank2}"].append(significance.pValue)
    
    cmi_ts_df = pd.DataFrame(cmi_series, index=dates)
    pval_ts_df = pd.DataFrame(pval_series, index=dates) if num_surrogates > 0 else None
    
    return (cmi_ts_df, pval_ts_df) if num_surrogates > 0 else cmi_ts_df


In [ ]:
# ============================================================================
# CALCULATE THEILER WINDOW AND VERIFY WITH ACF
# ============================================================================
# Set Theiler window
theiler_window = 0

# Visualize ACF to verify
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

max_lag = 50
all_variables = list(returns.columns)

for idx, col in enumerate(all_variables):
    ax = axes[idx]
    
    # Calculate ACF
    data = returns[col].values - np.mean(returns[col].values)
    lags = np.arange(0, min(max_lag, len(data) // 4))
    acf = np.zeros(len(lags))
    variance = np.var(data)
    
    for i, lag in enumerate(lags):
        acf[i] = 1.0 if lag == 0 else np.mean(data[:-lag] * data[lag:]) / variance
    
    # Plot ACF
    ax.stem(lags, acf, linefmt='C0-', markerfmt='C0o', basefmt='C0-')
    
    # Add significance threshold and Theiler window
    threshold = 2.0 / np.sqrt(len(data))
    ax.axhline(threshold, color='red', linestyle='--', linewidth=1, alpha=0.7, label='95% threshold')
    ax.axhline(-threshold, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

    
    ax.set_title(f'{col}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Lag (days)', fontsize=9)
    ax.set_ylabel('ACF', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.3, 1.1)
    
    if idx == 0:
        ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.suptitle(f'Autocorrelation Decay - Theiler Window = {theiler_window} days (Green Line)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.show()



In [ ]:
# ============================================================================
# CALCULATE MI AND CMI TIME SERIES
# ============================================================================

bank_names = list(BANKS.keys())
control_names = list(CONTROLS.keys())

# Calculate MI for all bank pairs with significance testing
mi_ts_df, mi_pvals = calculate_mi_time_series(
    returns, bank_names,
    WINDOW_SIZE, K_NEIGHBORS, theiler_window, ALGORITHM,
    num_surrogates=0  # 100 surrogates gives p-value accuracy to 0.01
)

# Calculate CMI conditioned on both control variables with significance testing
cmi_ts_df, cmi_pvals = calculate_cmi_time_series(
    returns, bank_names, control_names,
    WINDOW_SIZE, K_NEIGHBORS, theiler_window, ALGORITHM,
    num_surrogates=0  # 100 surrogates gives p-value accuracy to 0.01
)

# Calculate CMI conditioned on Interest Rates only (no significance testing for these)
cmi_interest_df = calculate_cmi_time_series(
    returns, bank_names, ['Interest_Rates'],
    WINDOW_SIZE, K_NEIGHBORS, theiler_window, ALGORITHM
)

# Calculate CMI conditioned on ASX200 only (no significance testing for these)
cmi_market_df = calculate_cmi_time_series(
    returns, bank_names, ['ASX200'],
    WINDOW_SIZE, K_NEIGHBORS, theiler_window, ALGORITHM
)

In [ ]:
# ============================================================================
# SAVE MI & CMI RESULTS TO CSV
# ============================================================================

# Save MI and CMI time series
mi_ts_df.to_csv("mi_time_series.csv", index=True)
cmi_ts_df.to_csv("cmi_both_controls.csv", index=True)
cmi_interest_df.to_csv("cmi_interest_rates.csv", index=True)
cmi_market_df.to_csv("cmi_asx200.csv", index=True)
mi_pvals.to_csv("mi_pvalues.csv", index=True)
cmi_pvals.to_csv("cmi_pvalues.csv", index=True)

In [ ]:
# ============================================================================
# Load MI & CMI RESULTS from CSV
# ============================================================================

mi_ts_df = pd.read_csv("mi_time_series.csv", index_col=0, parse_dates=True)
cmi_ts_df = pd.read_csv("cmi_both_controls.csv", index_col=0, parse_dates=True)
cmi_interest_df = pd.read_csv("cmi_interest_rates.csv", index_col=0, parse_dates=True)
cmi_market_df = pd.read_csv("cmi_asx200.csv", index_col=0, parse_dates=True)
mi_pvals = pd.read_csv("mi_pvalues.csv", index_col=0, parse_dates=True)
cmi_pvals = pd.read_csv("cmi_pvalues.csv", index_col=0, parse_dates=True)

# Visualisations

In [ ]:
# ============================================================================
# VISUALIZATION CONFIGURATION
# ============================================================================
# Define key market events for visualization across all graphs
# Format: 'YYYY-MM-DD': ('Event Label', 'color')

events = {
    '2015-06-01': ('China Growth Slowdown   ', 'black'),
    '2017-02-01': ('Trump 1st Term   ', 'black'),
    '2018-06-01': ('Banking Royal Commision   ', 'black'),
    '2020-03-01': ('COVID-19   ', 'black'),
    '2022-05-15': ('Inflation + RBA Rate Hikes   ', 'black'),
    '2023-03-01': ('SVB Crisis   ', 'black'),
    '2024-11-01': ('Trump 2nd Term                            ', 'black')
}

# Date range configuration for all time series plots
PLOT_START_DATE = '2015-01-01'  # Start from 1 Jan 2015

# X-axis formatting configuration
X_AXIS_INTERVAL = 6  # Show labels every 6 months
X_AXIS_FORMAT = '%b\n%Y'  # Format: Jan\n2015


In [ ]:
# ============================================================================
# VISUALIZATION: MI SIGNIFICANCE HEATMAP
# ============================================================================

# Create a combined visualization showing MI values with significance overlay
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Get pair columns (off-diagonal only) and sort by average MI (descending)
pair_columns = [col for col in mi_ts_df.columns if col.split('-')[0] != col.split('-')[1]]
avg_mi_per_pair = mi_ts_df[pair_columns].mean(axis=0).sort_values(ascending=False)
pair_columns_sorted = avg_mi_per_pair.index.tolist()

# Filter data from PLOT_START_DATE onwards
plot_start = pd.to_datetime(PLOT_START_DATE)
mi_data_filtered = mi_ts_df[mi_ts_df.index >= plot_start][pair_columns_sorted]
mi_pvals_filtered = mi_pvals[mi_pvals.index >= plot_start][pair_columns_sorted]

# Calculate shared scale for MI and CMI (will be used in both plots)
cmi_data_filtered = cmi_ts_df[cmi_ts_df.index >= plot_start][pair_columns_sorted]
shared_vmax = max(mi_data_filtered.values.max(), cmi_data_filtered.values.max())

# Top panel: MI values
ax = axes[0]
plot_data = mi_data_filtered.T
plot_dates = mi_data_filtered.index

im1 = ax.imshow(plot_data, aspect='auto', cmap='viridis', interpolation='nearest',
                vmin=0, vmax=shared_vmax)

ax.set_yticks(range(len(pair_columns_sorted)))
ax.set_yticklabels(pair_columns_sorted, fontsize=10, fontweight='bold')
ax.set_ylabel('Bank Pair', fontsize=12, fontweight='bold')
ax.set_title('Mutual Information Values (nats)', fontsize=13, fontweight='bold')

# Date ticks - 6 month intervals starting from January 2015
date_range = pd.date_range(start='2015-01-01', end=plot_dates[-1], freq='6MS')
date_positions = []
date_labels = []
for date in date_range:
    idx = np.argmin(np.abs(plot_dates - date))
    date_positions.append(idx)
    date_labels.append(date.strftime('%b\n%Y'))

ax.set_xticks(date_positions)
ax.set_xticklabels(date_labels, fontsize=9, rotation=0)

cbar1 = plt.colorbar(im1, ax=ax, pad=0.02)
cbar1.set_label('MI (nats)', fontsize=11, fontweight='bold')

# Bottom panel: Significance mask (p < 0.05)
ax = axes[1]
significance_mask = (mi_pvals_filtered < 0.05).astype(int).T

im2 = ax.imshow(significance_mask, aspect='auto', cmap='RdYlGn', interpolation='nearest',
                vmin=0, vmax=1)

ax.set_yticks(range(len(pair_columns_sorted)))
ax.set_yticklabels(pair_columns_sorted, fontsize=10, fontweight='bold')
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Bank Pair', fontsize=12, fontweight='bold')
ax.set_title('Statistical Significance (Green = Significant at p < 0.05)', fontsize=13, fontweight='bold')

ax.set_xticks(date_positions)
ax.set_xticklabels(date_labels, fontsize=9, rotation=0)

# Add colorbar for significance
cbar2 = plt.colorbar(im2, ax=ax, pad=0.02)
cbar2.set_label('Significance', fontsize=11, fontweight='bold')
cbar2.set_ticks([0, 1])

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# VISUALIZATION: CMI SIGNIFICANCE HEATMAP
# ============================================================================

# Create a combined visualization showing CMI values with significance overlay
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Get pair columns (off-diagonal only) and sort by average MI (same order as MI heatmap)
pair_columns = [col for col in cmi_ts_df.columns if col.split('-')[0] != col.split('-')[1]]
avg_mi_per_pair = mi_ts_df[pair_columns].mean(axis=0).sort_values(ascending=False)
pair_columns_sorted = avg_mi_per_pair.index.tolist()

# Filter data from PLOT_START_DATE onwards
plot_start = pd.to_datetime(PLOT_START_DATE)
cmi_data_filtered = cmi_ts_df[cmi_ts_df.index >= plot_start][pair_columns_sorted]
cmi_pvals_filtered = cmi_pvals[cmi_pvals.index >= plot_start][pair_columns_sorted]

# Use the same shared scale as MI heatmap for direct comparison
mi_data_filtered = mi_ts_df[mi_ts_df.index >= plot_start][pair_columns_sorted]
shared_vmax = max(mi_data_filtered.values.max(), cmi_data_filtered.values.max())

# Top panel: CMI values
ax = axes[0]
plot_data = cmi_data_filtered.T
plot_dates = cmi_data_filtered.index

im1 = ax.imshow(plot_data, aspect='auto', cmap='viridis', interpolation='nearest',
                vmin=0, vmax=shared_vmax)

ax.set_yticks(range(len(pair_columns_sorted)))
ax.set_yticklabels(pair_columns_sorted, fontsize=10, fontweight='bold')
ax.set_ylabel('Bank Pair', fontsize=12, fontweight='bold')
ax.set_title('Conditional Mutual Information Values (nats)', fontsize=13, fontweight='bold')

# Date ticks - 6 month intervals starting from January 2015
date_range = pd.date_range(start='2015-01-01', end=plot_dates[-1], freq='6MS')
date_positions = []
date_labels = []
for date in date_range:
    idx = np.argmin(np.abs(plot_dates - date))
    date_positions.append(idx)
    date_labels.append(date.strftime('%b\n%Y'))

ax.set_xticks(date_positions)
ax.set_xticklabels(date_labels, fontsize=9, rotation=0)

cbar1 = plt.colorbar(im1, ax=ax, pad=0.02)
cbar1.set_label('CMI (nats)', fontsize=11, fontweight='bold')

# Bottom panel: Significance mask (p < 0.05)
ax = axes[1]
significance_mask = (cmi_pvals_filtered < 0.05).astype(int).T

im2 = ax.imshow(significance_mask, aspect='auto', cmap='RdYlGn', interpolation='nearest',
                vmin=0, vmax=1)

ax.set_yticks(range(len(pair_columns_sorted)))
ax.set_yticklabels(pair_columns_sorted, fontsize=10, fontweight='bold')
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Bank Pair', fontsize=12, fontweight='bold')
ax.set_title('Statistical Significance (Green = Significant at p < 0.05)', fontsize=13, fontweight='bold')

ax.set_xticks(date_positions)
ax.set_xticklabels(date_labels, fontsize=9, rotation=0)

# Add colorbar for significance
cbar2 = plt.colorbar(im2, ax=ax, pad=0.02)
cbar2.set_label('Significance', fontsize=11, fontweight='bold')
cbar2.set_ticks([0, 1])

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# SIGNIFICANCE SUMMARY: PER BANK PAIR
# ============================================================================

# Calculate percentage of time each pair is significant for both MI and CMI
print("="*80)
print("SIGNIFICANCE SUMMARY BY BANK PAIR")
print("="*80)

# Get pair columns
pair_columns = [col for col in mi_ts_df.columns if col.split('-')[0] != col.split('-')[1]]

# Create summary DataFrame
summary_data = []
for pair in pair_columns:
    mi_sig_pct = 100 * (mi_pvals[pair] < 0.05).sum() / len(mi_pvals)
    cmi_sig_pct = 100 * (cmi_pvals[pair] < 0.05).sum() / len(cmi_pvals)
    
    mi_mean = mi_ts_df[pair].mean()
    cmi_mean = cmi_ts_df[pair].mean()
    
    summary_data.append({
        'Pair': pair,
        'MI Mean': mi_mean,
        'MI Sig%': mi_sig_pct,
        'CMI Mean': cmi_mean,
        'CMI Sig%': cmi_sig_pct,
        'Δ (MI-CMI)': mi_mean - cmi_mean
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('MI Sig%', ascending=False)

print("\nBank Pair Significance Summary (sorted by MI significance %)")
print("-"*80)
print(summary_df.to_string(index=False))
print("\nNote: 'Sig%' = percentage of time points with p < 0.05")
print("      'Δ (MI-CMI)' = information explained by market/interest rate controls")

In [ ]:
# ============================================================================
# AGGREGATE MARKET ANALYSIS: MEAN MI TIME SERIES
# ============================================================================

# For each bank, calculate the mean MI across all pairs involving that bank
bank_mi_ts = pd.DataFrame(index=mi_ts_df.index)

for bank in bank_names:
    # Find all columns where this bank appears in the pair name
    relevant_cols = [col for col in mi_ts_df.columns if bank in col.split('-')]
    # Calculate mean MI for this bank
    bank_mi_ts[bank] = mi_ts_df[relevant_cols].mean(axis=1)


# Calculate overall mean MI across all bank pairs over time
mean_mi_market = bank_mi_ts.mean(axis=1)

# Filter data from PLOT_START_DATE onwards
plot_start = pd.to_datetime(PLOT_START_DATE)
mean_mi_market_filtered = mean_mi_market[mean_mi_market.index >= plot_start]

# Create line plot
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(mean_mi_market_filtered.index, mean_mi_market_filtered.values, color='blue', linewidth=2)
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Mutual Information', fontsize=12, fontweight='bold')
ax.set_title('Mean Mutual Information for Australian Banking System', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Set x-axis limits and formatting - use January and July
ax.set_xlim(mean_mi_market_filtered.index[0], mean_mi_market_filtered.index[-1])
# Create dates for January and July of each year
years = pd.date_range(start=mean_mi_market_filtered.index[0].replace(month=1, day=1),
                      end=mean_mi_market_filtered.index[-1], freq='6MS')
ax.set_xticks(years)
ax.xaxis.set_major_formatter(mdates.DateFormatter(X_AXIS_FORMAT))

# Add crisis event markers with text labels
for date_str, (label, color) in events.items():
    event_date = pd.to_datetime(date_str)
    if event_date >= mean_mi_market_filtered.index[0] and event_date <= mean_mi_market_filtered.index[-1]:
        ax.axvline(event_date, color=color, linestyle='--', linewidth=1.5, alpha=0.6)
        ax.text(event_date, ax.get_ylim()[1], label, rotation=90, 
                fontsize=9, color=color, ha='right', va='top', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# SEPARATE CONDITIONING EFFECTS
# ============================================================================

# Calculate mean CMI across all pairs for each conditioning scenario
# Note: Using pre-calculated cmi_interest_df and cmi_market_df from the calculation cell
mean_mi = mi_ts_df[[col for col in mi_ts_df.columns if col.split('-')[0] != col.split('-')[1]]].mean(axis=1)
mean_cmi_interest = cmi_interest_df.mean(axis=1)
mean_cmi_market = cmi_market_df.mean(axis=1)
mean_cmi_both = cmi_ts_df.mean(axis=1)

# Filter data from PLOT_START_DATE onwards
plot_start = pd.to_datetime(PLOT_START_DATE)
mean_mi_filtered = mean_mi[mean_mi.index >= plot_start]
mean_cmi_interest_filtered = mean_cmi_interest[mean_cmi_interest.index >= plot_start]
mean_cmi_market_filtered = mean_cmi_market[mean_cmi_market.index >= plot_start]
mean_cmi_both_filtered = mean_cmi_both[mean_cmi_both.index >= plot_start]

# Calculate uncertainty reduction (MI - CMI)
uncertainty_reduction_interest = mean_mi_filtered - mean_cmi_interest_filtered
uncertainty_reduction_market = mean_mi_filtered - mean_cmi_market_filtered
uncertainty_reduction_both = mean_mi_filtered - mean_cmi_both_filtered

# Create visualization - single panel showing CMI for different conditioning scenarios
fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(mean_mi_filtered.index, mean_mi_filtered.values, color='black', linewidth=2, label='MI', linestyle='-')
ax.plot(mean_cmi_interest_filtered.index, mean_cmi_interest_filtered.values, color='blue', linewidth=2, label='CMI | Aus. Govt. Bond Index')
ax.plot(mean_cmi_market_filtered.index, mean_cmi_market_filtered.values, color='green', linewidth=2, label='CMI | ASX200 Ex-T20 Index')
ax.plot(mean_cmi_both_filtered.index, mean_cmi_both_filtered.values, color='red', linewidth=2, label='CMI | Both Variables')
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Mutual Information (nats)', fontsize=12, fontweight='bold')
ax.set_title('Mean Mutual Information Across Conditioning Factors', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)

# Set x-axis limits and formatting - use January and July
ax.set_xlim(mean_mi_filtered.index[0], mean_mi_filtered.index[-1])
# Create dates for January and July of each year
years = pd.date_range(start=mean_mi_filtered.index[0].replace(month=1, day=1),
                      end=mean_mi_filtered.index[-1], freq='6MS')
ax.set_xticks(years)
ax.xaxis.set_major_formatter(mdates.DateFormatter(X_AXIS_FORMAT))

# Add crisis event markers with text labels
for date_str, (label, color) in events.items():
    event_date = pd.to_datetime(date_str)
    if event_date >= mean_mi_filtered.index[0] and event_date <= mean_mi_filtered.index[-1]:
        ax.axvline(event_date, color=color, linestyle='--', linewidth=1.2, alpha=0.4)
        ax.text(event_date, ax.get_ylim()[1], label, rotation=90, 
                fontsize=11, color=color, ha='right', va='top', fontweight='bold')

plt.tight_layout()
plt.show()


# Robustness Checks

In [ ]:
# ============================================================================
# PEARSON CORRELATION ANALYSIS WITH 150-DAY ROLLING WINDOW
# ============================================================================

# Calculate rolling Pearson correlations for all bank pairs
window_size = 150
bank_names = list(BANKS.keys())

# Initialize storage for correlation time series
corr_series = {f"{b1}-{b2}": [] for i, b1 in enumerate(bank_names) 
               for j, b2 in enumerate(bank_names) if i < j}
dates = []

# Sliding window correlation calculation
for end_idx in range(window_size, len(returns) + 1):
    window_data = returns.iloc[end_idx - window_size:end_idx]
    dates.append(returns.index[end_idx - 1])
    
    # Calculate correlation for all bank pairs
    for i, bank1 in enumerate(bank_names):
        for j, bank2 in enumerate(bank_names):
            if i < j:
                corr_value = window_data[bank1].corr(window_data[bank2])
                corr_series[f"{bank1}-{bank2}"].append(corr_value)

# Create DataFrame
corr_ts_df = pd.DataFrame(corr_series, index=dates)

print(f"✓ Calculated rolling Pearson correlations")
print(f"  Window size: {window_size} days")
print(f"  Number of bank pairs: {len(corr_series)}")
print(f"  Date range: {corr_ts_df.index[0].strftime('%Y-%m-%d')} to {corr_ts_df.index[-1].strftime('%Y-%m-%d')}")

# ============================================================================
# VISUALIZATION: PEARSON CORRELATION HEATMAP
# ============================================================================

# Create visualization showing correlation values
fig, ax = plt.subplots(1, 1, figsize=(16, 10))

# Get pair columns and sort by average MI (same order as MI/CMI heatmaps)
pair_columns = [col for col in corr_ts_df.columns if col.split('-')[0] != col.split('-')[1]]
avg_mi_per_pair = mi_ts_df[pair_columns].mean(axis=0).sort_values(ascending=False)
pair_columns_sorted = avg_mi_per_pair.index.tolist()

# Filter data from PLOT_START_DATE onwards
plot_start = pd.to_datetime(PLOT_START_DATE)
corr_data_filtered = corr_ts_df[corr_ts_df.index >= plot_start][pair_columns_sorted]

# Plot correlation values
plot_data = corr_data_filtered.T
plot_dates = corr_data_filtered.index

im = ax.imshow(plot_data, aspect='auto', cmap='RdYlBu_r', interpolation='nearest',
               vmin=-1, vmax=1)

ax.set_yticks(range(len(pair_columns_sorted)))
ax.set_yticklabels(pair_columns_sorted, fontsize=10, fontweight='bold')
ax.set_ylabel('Bank Pair', fontsize=12, fontweight='bold')
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_title('Pearson Correlation Coefficients Over Time (150-day Rolling Window)', 
             fontsize=13, fontweight='bold')

# Date ticks - 6 month intervals starting from January 2015
date_range = pd.date_range(start='2015-01-01', end=plot_dates[-1], freq='6MS')
date_positions = []
date_labels = []
for date in date_range:
    idx = np.argmin(np.abs(plot_dates - date))
    date_positions.append(idx)
    date_labels.append(date.strftime('%b\n%Y'))

ax.set_xticks(date_positions)
ax.set_xticklabels(date_labels, fontsize=9, rotation=0)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, pad=0.02)
cbar.set_label('Pearson Correlation', fontsize=11, fontweight='bold')

plt.suptitle('Pearson Correlation: Bank Pairs Sorted by Average MI', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n{'='*80}")
print("Pearson Correlation Summary Statistics:")
print(f"{'='*80}")
print(f"Mean correlation across all pairs/time: {corr_data_filtered.values.mean():.4f}")
print(f"Min correlation: {corr_data_filtered.values.min():.4f}")
print(f"Max correlation: {corr_data_filtered.values.max():.4f}")
print(f"Std deviation: {corr_data_filtered.values.std():.4f}")


In [ ]:
# ============================================================================
# WINDOW SIZE COMPARISON: MEAN MI WITH DIFFERENT WINDOW SIZES
# ============================================================================

print("Calculating Mean MI with different window sizes...")
print("This will take several minutes...\n")

window_sizes = [50, 100, 150, 200, 300]
mean_mi_by_window = {}

for ws in window_sizes:
    print(f"Calculating with window size = {ws} days...")
    
    # Calculate MI time series with current window size (no significance testing for speed)
    mi_ts_temp = calculate_mi_time_series(
        returns, bank_names,
        ws, K_NEIGHBORS, theiler_window, ALGORITHM
    )
    
    # Calculate mean MI for each bank
    bank_mi_temp = pd.DataFrame(index=mi_ts_temp.index)
    for bank in bank_names:
        relevant_cols = [col for col in mi_ts_temp.columns if bank in col.split('-')]
        bank_mi_temp[bank] = mi_ts_temp[relevant_cols].mean(axis=1)
    
    # Calculate overall mean across all banks and filter from PLOT_START_DATE
    plot_start = pd.to_datetime(PLOT_START_DATE)
    mean_mi_temp = bank_mi_temp.mean(axis=1)
    mean_mi_by_window[ws] = mean_mi_temp[mean_mi_temp.index >= plot_start]
    print(f"  ✓ Complete. Range: {mean_mi_by_window[ws].min():.4f} to {mean_mi_by_window[ws].max():.4f} nats")

# Create comparison plot
fig, ax = plt.subplots(figsize=(16, 8))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for idx, ws in enumerate(window_sizes):
    ax.plot(mean_mi_by_window[ws].index, mean_mi_by_window[ws].values, 
            color=colors[idx], linewidth=2, label=f'{ws} days', alpha=0.8)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Mutual Information (nats)', fontsize=12, fontweight='bold')
ax.set_title('Mean MI for Australian Banking System: Window Size Comparison', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=11, framealpha=0.9, title='Window Size')

# Set x-axis limits and formatting - use January and July
ax.set_xlim(mean_mi_by_window[300].index[0], mean_mi_by_window[300].index[-1])
# Create dates for January and July of each year
years = pd.date_range(start=mean_mi_by_window[300].index[0].replace(month=1, day=1),
                      end=mean_mi_by_window[300].index[-1], freq='6MS')
ax.set_xticks(years)
ax.xaxis.set_major_formatter(mdates.DateFormatter(X_AXIS_FORMAT))

# Add crisis event markers using global events dictionary
for date_str, (label, color) in events.items():
    event_date = pd.to_datetime(date_str)
    # Check against the longest window size (300) for consistency
    if event_date >= mean_mi_by_window[300].index[0] and event_date <= mean_mi_by_window[300].index[-1]:
        ax.axvline(event_date, color=color, linestyle='--', linewidth=1.5, alpha=0.4)
        ax.text(event_date, ax.get_ylim()[1], label, rotation=90, 
                fontsize=8, color=color, ha='right', va='top', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n{'='*80}")
print("Window Size Statistics:")
print(f"{'='*80}")
for ws in window_sizes:
    print(f"Window = {ws:3d} days: Mean = {mean_mi_by_window[ws].mean():.4f}, "
          f"Min = {mean_mi_by_window[ws].min():.4f}, Max = {mean_mi_by_window[ws].max():.4f}")
print(f"\nDate range: {mean_mi_by_window[300].index[0].strftime('%Y-%m-%d')} to {mean_mi_by_window[300].index[-1].strftime('%Y-%m-%d')}")

In [ ]:
# ============================================================================
# K NEIGHBORS COMPARISON: MEAN MI WITH DIFFERENT K NEIGHBORS
# ============================================================================

print("Calculating Mean MI with different k_neighbors values...")
print("This will take several minutes...\n")

k_neighbors_list = [2, 4, 6, 8, 10]
mean_mi_by_k = {}

for k in k_neighbors_list:
    print(f"Calculating with k_neighbors = {k}...")
    
    # Calculate MI time series with current k_neighbors (no significance testing for speed)
    mi_ts_temp = calculate_mi_time_series(
        returns, bank_names,
        WINDOW_SIZE, k, theiler_window, ALGORITHM
    )
    
    # Calculate mean MI for each bank
    bank_mi_temp = pd.DataFrame(index=mi_ts_temp.index)
    for bank in bank_names:
        relevant_cols = [col for col in mi_ts_temp.columns if bank in col.split('-')]
        bank_mi_temp[bank] = mi_ts_temp[relevant_cols].mean(axis=1)
    
    # Calculate overall mean across all banks and filter from PLOT_START_DATE
    plot_start = pd.to_datetime(PLOT_START_DATE)
    mean_mi_temp = bank_mi_temp.mean(axis=1)
    mean_mi_by_k[k] = mean_mi_temp[mean_mi_temp.index >= plot_start]
    print(f"  ✓ Complete. Range: {mean_mi_by_k[k].min():.4f} to {mean_mi_by_k[k].max():.4f} nats")

# Create comparison plot
fig, ax = plt.subplots(figsize=(16, 8))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for idx, k in enumerate(k_neighbors_list):
    ax.plot(mean_mi_by_k[k].index, mean_mi_by_k[k].values, 
            color=colors[idx], linewidth=2, label=f'k = {k}', alpha=0.8)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Mutual Information (nats)', fontsize=12, fontweight='bold')
ax.set_title('Mean MI for Australian Banking System: k_neighbors Comparison', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=11, framealpha=0.9, title='k_neighbors')

# Set x-axis limits and formatting - use January and July
ax.set_xlim(mean_mi_by_k[k_neighbors_list[0]].index[0], mean_mi_by_k[k_neighbors_list[0]].index[-1])
# Create dates for January and July of each year
years = pd.date_range(start=mean_mi_by_k[k_neighbors_list[0]].index[0].replace(month=1, day=1),
                      end=mean_mi_by_k[k_neighbors_list[0]].index[-1], freq='6MS')
ax.set_xticks(years)
ax.xaxis.set_major_formatter(mdates.DateFormatter(X_AXIS_FORMAT))

# Add crisis event markers using global events dictionary
for date_str, (label, color) in events.items():
    event_date = pd.to_datetime(date_str)
    # Use the first k value to check date range
    if event_date >= mean_mi_by_k[k_neighbors_list[0]].index[0] and event_date <= mean_mi_by_k[k_neighbors_list[0]].index[-1]:
        ax.axvline(event_date, color=color, linestyle='--', linewidth=1.5, alpha=0.4)
        ax.text(event_date, ax.get_ylim()[1], label, rotation=90, 
                fontsize=8, color=color, ha='right', va='top', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n{'='*80}")
print("k_neighbors Statistics:")
print(f"{'='*80}")
for k in k_neighbors_list:
    print(f"k = {k:2d}: Mean = {mean_mi_by_k[k].mean():.4f}, "
          f"Min = {mean_mi_by_k[k].min():.4f}, Max = {mean_mi_by_k[k].max():.4f}")
print(f"\nDate range: {mean_mi_by_k[k_neighbors_list[0]].index[0].strftime('%Y-%m-%d')} to {mean_mi_by_k[k_neighbors_list[0]].index[-1].strftime('%Y-%m-%d')}")